## 1. Cai Dat Dependencies

In [ ]:
import torch;
print(torch.__version__);
print(torch.version.cuda);
print(torch.cuda.is_available());
print(torch.cuda.get_device_name(0)) if torch.cuda.is_available() else "No GPU available"

In [ ]:
# Cài đặt các thư viện cần thiết
# %pip install torch torchvision transformers accelerate pillow einops timm gradio sentencepiece safetensors requests matplotlib
# %pip install peft
# %pip install scikit-learn

In [ ]:
import torch
import torch.nn as nn
from PIL import Image
from transformers import AutoModel, AutoModelForCausalLM, AutoImageProcessor, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from dataclasses import dataclass
from typing import Optional, List
import requests
from io import BytesIO


In [ ]:
# ============================================
# GLOBAL CONFIGURATION - current notebook contract
# ============================================

import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    COMPUTE_DTYPE = torch.bfloat16
else:
    COMPUTE_DTYPE = torch.float16

VISION_MODEL_NAME = "microsoft/swinv2-base-patch4-window8-256"
VISION_HIDDEN_SIZE = 1024
LLM_MODEL_NAME = "internlm/internlm2_5-1_8b"
LLM_HIDDEN_SIZE = 2048

IMAGE_SIZE = 256
MAX_TEXT_LENGTH = 256
MAX_IMAGES = 5
TOP_K_IMAGES = 3

DATA_DIR = "datasets"
IMAGE_DIR = os.path.join(DATA_DIR, "image")
OUTPUT_DIR = "output_model"
BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, "best_model.safetensors")
TRAINING_INFO_PATH = os.path.join(OUTPUT_DIR, "training_info.pt")

BATCH_SIZE = 8
NUM_WORKERS = 0

print(f"Compute dtype: {COMPUTE_DTYPE}")
print(f"Vision model: {VISION_MODEL_NAME}")
print(f"LLM model:    {LLM_MODEL_NAME}")
print(f"Data dir:     {DATA_DIR}")
print(f"Output dir:   {OUTPUT_DIR}")
print(f"Image size:   {IMAGE_SIZE}")
print(f"MAX_IMAGES={MAX_IMAGES}, TOP_K_IMAGES={TOP_K_IMAGES}")


---
## 1.5. Demo Dataset Structure

Xem cau truc du lieu truoc khi bat dau.

In [ ]:
# ============================================
# DEMO DATASET STRUCTURE
# ============================================

import json
from collections import Counter
from PIL import Image
import matplotlib.pyplot as plt

with open(os.path.join(DATA_DIR, "train.json"), "r", encoding="utf-8") as f:
    train_data = json.load(f)

print(f"Total training samples: {len(train_data)}")
print("\n" + "=" * 80)
print("SAMPLE DATA STRUCTURE:")
print("=" * 80)

sample = train_data[0]
print(f"\nSample keys: {list(sample.keys())}")
print(f"\nComment: {sample.get('comment', 'N/A')}")
print(f"\nImage files: {sample.get('list_img', [])}")
print(f"\nLabels (text_img_label): {sample.get('text_img_label', [])}")

if sample.get("list_img") and len(sample["list_img"]) > 0:
    img_path = os.path.join(IMAGE_DIR, sample["list_img"][0])
    if os.path.exists(img_path):
        print(f"\nImage path: {img_path}")
        img = Image.open(img_path)
        plt.figure(figsize=(8, 6))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"Sample Image\nLabels: {sample.get('text_img_label', [])}")
        plt.tight_layout()
        plt.show()

        print(f"\nImage size: {img.size}")
        print(f"Image mode: {img.mode}")
    else:
        print(f"\n[WARNING] Image not found: {img_path}")

print("\n" + "=" * 80)
print("LABEL DISTRIBUTION:")
print("=" * 80)

all_labels = []
for item in train_data:
    all_labels.extend(item.get("text_img_label", []))

label_counts = Counter(all_labels)
print(f"\nTotal labels: {len(all_labels)}")
print(f"Unique labels: {len(label_counts)}")
print("\nTop 10 most common labels:")
for label, count in label_counts.most_common(10):
    print(f"  {label:<40} {count:>5} ({count/len(all_labels)*100:.1f}%)")


---
## 2. Preprocessor và Swin Transformer V2 (Vision Encoder)

In [ ]:
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

# ============================================
# PREPROCESSOR + SWIN V2 VISION ENCODER
# ============================================

swin_image_processor = AutoImageProcessor.from_pretrained(VISION_MODEL_NAME)

def _ensure_rgb(img):
    return img.convert('RGB') if img.mode != 'RGB' else img

def build_transform(input_size=IMAGE_SIZE):
    """Evaluation/test transform using SwinV2 AutoImageProcessor."""
    def _transform(img):
        img = _ensure_rgb(img)
        proc_inputs = swin_image_processor(
            images=img,
            return_tensors='pt',
            do_resize=True,
            size={'height': input_size, 'width': input_size},
            do_center_crop=False,
        )
        return proc_inputs['pixel_values'].squeeze(0)
    return _transform

def build_train_transform(input_size=IMAGE_SIZE):
    """Training transform with light augmentation + SwinV2 AutoImageProcessor."""
    train_aug = T.Compose([
        T.RandomResizedCrop(
            (input_size, input_size),
            scale=(0.85, 1.0),
            interpolation=InterpolationMode.BICUBIC,
        ),
        T.RandomHorizontalFlip(p=0.5),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    ])

    def _transform(img):
        img = _ensure_rgb(img)
        img = train_aug(img)
        proc_inputs = swin_image_processor(
            images=img,
            return_tensors='pt',
            do_resize=False,
            do_center_crop=False,
        )
        return proc_inputs['pixel_values'].squeeze(0)

    return _transform

# ============================================
# SWIN TRANSFORMER V2 - Vision Encoder
# ============================================

class VisionEncoder(nn.Module):
    """
    Swin Transformer V2 vision encoder.

    Input: [B, 3, IMAGE_SIZE, IMAGE_SIZE]
    Output: [B, num_patches, VISION_HIDDEN_SIZE]
    """

    def __init__(
        self,
        model_name: str = VISION_MODEL_NAME,
        device: str = device,
        torch_dtype: torch.dtype = COMPUTE_DTYPE,
    ):
        super().__init__()
        self.device = device
        self.torch_dtype = torch_dtype

        print(f"Loading Swin Transformer V2: {model_name}")
        self.model = AutoModel.from_pretrained(model_name, torch_dtype=torch_dtype).to(device)

        for param in self.model.parameters():
            param.requires_grad = False
        self.model.eval()

        self.hidden_size = self.model.config.hidden_size
        with torch.no_grad():
            dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, dtype=torch_dtype, device=device)
            dummy_out = self.model(pixel_values=dummy).last_hidden_state
            self.num_patches = dummy_out.shape[1]

        print(f"Loaded: hidden_size={self.hidden_size}, num_patches={self.num_patches} (auto-detected)")

    @torch.no_grad()
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """Extract Swin image tokens."""
        pixel_values = pixel_values.to(self.device, dtype=self.torch_dtype)
        outputs = self.model(pixel_values=pixel_values)
        return outputs.last_hidden_state


---
## 3. Projection Layer (MLP)

In [ ]:
class MLPProjector(nn.Module):
    """
    Project Swin image tokens into the InternLM2.5 embedding space.

    Input: [B, num_patches, VISION_HIDDEN_SIZE]
    Output: [B, num_patches, LLM_HIDDEN_SIZE]

    """

    def __init__(
        self,
        vision_dim: int = VISION_HIDDEN_SIZE,
        llm_dim: int = LLM_HIDDEN_SIZE,
    ):
        super().__init__()
        self.vision_dim = vision_dim
        self.llm_dim = llm_dim

        self.projector = nn.Sequential(
            nn.LayerNorm(vision_dim),
            nn.Linear(vision_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
        )

        for module in self.projector:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)


    def forward(self, img_tokens: torch.Tensor) -> torch.Tensor:
        """Project image tokens to language-token embeddings."""
        img_tokens = img_tokens.to(dtype=self.projector[0].weight.dtype)
        return self.projector(img_tokens)


In [ ]:
# ============================================
# KHOI TAO SWIN V2 VISION ENCODER VA PROJECTOR
# ============================================

vision_encoder = VisionEncoder(
    model_name=VISION_MODEL_NAME,
    device=device,
    torch_dtype=COMPUTE_DTYPE,
)

projector = MLPProjector(
    vision_dim=VISION_HIDDEN_SIZE,
    llm_dim=LLM_HIDDEN_SIZE,
).to(device)

print(f"  Vision model: {VISION_MODEL_NAME}")
print(f"  Vision hidden dim: {vision_encoder.hidden_size}")
print(f"  Vision num patches: {vision_encoder.num_patches}")
print(f"  Target LLM hidden dim: {projector.llm_dim}")
print(f"  Projector trainable params: {sum(p.numel() for p in projector.parameters()):,}")


---
## 4. LLM - InternLM2.5-1.8B


In [ ]:
# ============================================
# LOAD LLM - InternLM2.5-1.8B
# ============================================

MODEL_NAME = LLM_MODEL_NAME

print(f"Loading {MODEL_NAME}...")
print("Neu model gated, can dang nhap Hugging Face token truoc khi load.")

torch_dtype = COMPUTE_DTYPE

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    if tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token
    elif tokenizer.unk_token_id is not None:
        tokenizer.pad_token = tokenizer.unk_token
    else:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})

tokenizer.padding_side = "right"
print(f"Tokenizer padding_side: {tokenizer.padding_side}")
print(f"Tokenizer pad_token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch_dtype,
    attn_implementation="eager",
).to(device).eval()

print("Model loaded!")
print(f"  Model type: {type(model).__name__}")
print(f"  Hidden size: {model.config.hidden_size}")
print(f"  Num layers: {model.config.num_hidden_layers}")
print(f"  Dtype: {torch_dtype}")

if model.config.hidden_size != LLM_HIDDEN_SIZE:
    print(f"[WARNING] LLM hidden_size ({model.config.hidden_size}) != LLM_HIDDEN_SIZE ({LLM_HIDDEN_SIZE})")
    print("          Hay cap nhat LLM_HIDDEN_SIZE de khop model ban dang dung.")


---
## 5. Dataset va DataLoader

In [ ]:
import json
import os
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ============================================
# LABEL DEFINITIONS - 4-class labels for two-stage training
# ============================================

ASPECT_LABELS = ["Facilities", "Public_area", "Location", "Food", "Room", "Service"]
ASPECT2ID = {label: idx for idx, label in enumerate(ASPECT_LABELS)}
ID2ASPECT = {idx: label for idx, label in enumerate(ASPECT_LABELS)}
NUM_ASPECTS = len(ASPECT_LABELS)

CLASS_LABELS = ["None", "Negative", "Neutral", "Positive"]
CLASS2ID = {label: idx for idx, label in enumerate(CLASS_LABELS)}
ID2CLASS = {idx: label for idx, label in enumerate(CLASS_LABELS)}
NUM_CLASSES = len(CLASS_LABELS)

_SENTIMENT_TO_CLASS = {"Negative": 1, "Neutral": 2, "Positive": 3}

print(f"Aspect Categories ({NUM_ASPECTS}): {ASPECT_LABELS}")
print(f"Classes per Aspect ({NUM_CLASSES}): {CLASS_LABELS}")
print(f"Total output dim: {NUM_ASPECTS} x {NUM_CLASSES} = {NUM_ASPECTS * NUM_CLASSES}")


In [ ]:
# ============================================
# DATA LOADER - Load train/dev/test JSON splits
# ============================================

def load_dataset_json(json_path: str) -> list:
    """Doc file JSON dataset."""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"Loaded {len(data)} samples from {json_path}")
    return data


def load_all_splits(data_dir: str = DATA_DIR):
    """Load all dataset splits from the configured data directory."""
    splits = {}
    for split in ['train', 'dev', 'test']:
        json_path = os.path.join(data_dir, f'{split}.json')
        if os.path.exists(json_path):
            splits[split] = load_dataset_json(json_path)
        else:
            print(f"[WARNING] {json_path} not found!")
            splits[split] = []
    return splits


dataset_splits = load_all_splits(DATA_DIR)

print(f"\nDataset Statistics:")
print(f"  Train: {len(dataset_splits['train'])} samples")
print(f"  Dev:   {len(dataset_splits['dev'])} samples")
print(f"  Test:  {len(dataset_splits['test'])} samples")


In [ ]:
# ============================================
# PYTORCH DATASET CLASS - Two-Stage Label Contract
# Multi-image support, collate_fn uses closure
# ============================================

class SentimentDataset(Dataset):
    """
    PyTorch Dataset cho Aspect-Category Sentiment Analysis.

    Moi sample gom:
    - comment: Text binh luan
    - image_paths: TAT CA valid image paths (toi da MAX_IMAGES)
    - labels: [6] int tensor theo 4-class contract per aspect
        0 = None (aspect khong xuat hien)
        1 = Negative
        2 = Neutral
        3 = Positive

    Two-stage model se tach label nay thanh:
    - Presence target: label > 0
    - Sentiment target: label - 1 cho cac aspect present
    """

    def __init__(
        self,
        data: list,
        image_dir: str,
        aspect2id: dict,
        transform=None,
    ):
        self.data = data
        self.image_dir = image_dir
        self.aspect2id = aspect2id
        self.num_aspects = len(aspect2id)
        self.transform = transform if transform else build_transform(IMAGE_SIZE)

        self.samples = self._prepare_samples()
        print(f"Prepared {len(self.samples)} valid samples")

    def _parse_label(self, label: str):
        """
        Parse label goc thanh (aspect, sentiment).
        VD: 'Room#Positive' -> ('Room', 'Positive')
        Returns None neu label khong hop le.
        """
        if '#' not in label:
            return None
        parts = label.split('#')
        if len(parts) != 2:
            return None
        aspect, sentiment = parts[0], parts[1]
        if aspect in self.aspect2id and sentiment in _SENTIMENT_TO_CLASS:
            return (aspect, sentiment)
        return None

    def _prepare_samples(self) -> list:
        """Loc va chuan bi samples hop le. Giu TAT CA anh hop le (toi da MAX_IMAGES)."""
        valid_samples = []

        for item in self.data:
            if not item.get('list_img') or len(item['list_img']) == 0:
                continue

            raw_labels = item.get('text_img_label', [])
            if not raw_labels:
                continue

            parsed_labels = []
            for label in raw_labels:
                parsed = self._parse_label(label)
                if parsed is not None:
                    parsed_labels.append(parsed)

            if not parsed_labels:
                continue

            valid_img_paths = []
            for img_name in item['list_img'][:MAX_IMAGES]:
                img_path = os.path.join(self.image_dir, img_name)
                if os.path.exists(img_path):
                    valid_img_paths.append(img_path)

            if not valid_img_paths:
                continue

            valid_samples.append({
                'comment': item.get('comment', ''),
                'image_paths': valid_img_paths,
                'parsed_labels': parsed_labels,
                'raw_labels': raw_labels,
            })

        return valid_samples

    def __len__(self):
        return len(self.samples)

    def _labels_to_tensor(self, parsed_labels: list) -> torch.Tensor:
        """
        Convert list of (aspect, sentiment) thanh compact 4-class tensor [num_aspects].

        0 = None, 1 = Negative, 2 = Neutral, 3 = Positive.
        """
        labels = torch.zeros(self.num_aspects, dtype=torch.long)

        for aspect, sentiment in parsed_labels:
            a_id = self.aspect2id[aspect]
            labels[a_id] = _SENTIMENT_TO_CLASS[sentiment]

        return labels

    def __getitem__(self, idx: int) -> dict:
        sample = self.samples[idx]

        image_tensors = []
        for img_path in sample['image_paths']:
            image = Image.open(img_path).convert('RGB')
            image_tensors.append(self.transform(image))

        pixel_values = torch.stack(image_tensors)
        labels = self._labels_to_tensor(sample['parsed_labels'])

        return {
            'pixel_values': pixel_values,
            'num_images': len(image_tensors),
            'labels': labels,
            'comment': sample['comment'],
            'image_paths': sample['image_paths'],
        }


def make_collate_fn(tokenizer_ref):
    """Tao collate function voi tokenizer reference (closure)."""
    def collate_fn(batch):
        image_counts = [item['num_images'] for item in batch]
        max_imgs = max(image_counts)

        padded_pixels = []
        for item in batch:
            pvs = item['pixel_values']
            n = pvs.shape[0]
            if n < max_imgs:
                pad = torch.zeros(max_imgs - n, *pvs.shape[1:], dtype=pvs.dtype)
                padded_pixels.append(torch.cat([pvs, pad], dim=0))
            else:
                padded_pixels.append(pvs)

        pixel_values = torch.stack(padded_pixels)
        image_counts_tensor = torch.tensor(image_counts)

        labels = torch.stack([item['labels'] for item in batch])
        comments = [item['comment'] for item in batch]
        image_paths = [item['image_paths'] for item in batch]

        text_inputs = tokenizer_ref(
            comments,
            padding=True,
            truncation=True,
            max_length=MAX_TEXT_LENGTH,
            return_tensors='pt'
        )

        return {
            'pixel_values': pixel_values,
            'image_counts': image_counts_tensor,
            'input_ids': text_inputs.input_ids,
            'attention_mask': text_inputs.attention_mask,
            'labels': labels,
            'comments': comments,
            'image_paths': image_paths,
        }

    return collate_fn


In [ ]:
# ============================================
# TAO DATALOADERS (Two-Stage Labels, Multi-Image, Augmented)
# ============================================

train_dataset = SentimentDataset(
    data=dataset_splits['train'],
    image_dir=IMAGE_DIR,
    aspect2id=ASPECT2ID,
    transform=build_train_transform(IMAGE_SIZE),
)

dev_dataset = SentimentDataset(
    data=dataset_splits['dev'],
    image_dir=IMAGE_DIR,
    aspect2id=ASPECT2ID,
    transform=build_transform(IMAGE_SIZE),
)

test_dataset = SentimentDataset(
    data=dataset_splits['test'],
    image_dir=IMAGE_DIR,
    aspect2id=ASPECT2ID,
    transform=build_transform(IMAGE_SIZE),
)

collate_fn = make_collate_fn(tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
)

print(f"\nDataLoader Statistics:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Dev batches:   {len(dev_loader)}")
print(f"  Test batches:  {len(test_loader)}")

sample_batch = next(iter(train_loader))
print(f"\nSample batch:")
print(f"  pixel_values shape: {sample_batch['pixel_values'].shape}")
print(f"  image_counts:       {sample_batch['image_counts'].tolist()}")
print(f"  labels shape:       {sample_batch['labels'].shape}")
print(f"  labels sample:      {sample_batch['labels'][0].tolist()}")


---
## 6. Training Loop


In [ ]:
# Suppress transformers deprecation warning that causes logging error
import warnings
warnings.filterwarnings("ignore", message=".*AttentionMaskConverter.*")
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

# ============================================
# MULTIMODAL SENTIMENT MODEL — REFACTORED
# Architecture:
#   Swin V2 → MLPProjector → Top-K image selection (K = TOP_K_IMAGES)
#   → concat [K*P image tokens | text tokens]
#   → InternLM2.5 (LoRA) → full hidden states H [B, L, H_llm]
#   → 6 aspect queries cross-attend into H → Z [B, 6, H_llm]
#   → presence_head(Z) → [B, 6] (aspect present?)
#   → sentiment_head(Z) → [B, 6, 3] (Neg/Neu/Pos)
# ============================================

class MultimodalSentimentModel(nn.Module):
    """
    Two-Stage Aspect-Query Sentiment Model.

    Stage 1 — Presence: Is aspect i mentioned? (binary per aspect)
    Stage 2 — Sentiment: If present, what sentiment? (3-class: Neg/Neu/Pos)

    Key improvements over v1:
      - 6 learnable aspect queries cross-attend into ALL LLM hidden states
        (replaces single h_last → legacy single-head classifier)
      - Top-K image selection preserves per-image evidence
        (replaces weighted-sum that collapsed all images into one)
      - Two-stage classification separates "is aspect present?"
        from "what sentiment?" (replaces 4-class softmax with None)
    """

    def __init__(
        self,
        vision_encoder: VisionEncoder,
        projector: MLPProjector,
        llm_model,
        tokenizer,
        num_aspects: int = 6,
        num_sentiment_classes: int = 3,  # Neg/Neu/Pos (no None!)
        top_k_images: int = TOP_K_IMAGES,
        lora_r: int = 16,
        lora_alpha: int = 16,
        lora_dropout: float = 0.0,
    ):
        super().__init__()

        self.vision_encoder = vision_encoder
        self.projector = projector
        self.tokenizer = tokenizer
        self.num_aspects = num_aspects
        self.num_sentiment_classes = num_sentiment_classes
        self.top_k_images = top_k_images
        self.fusion_mode = "top_k_image_selection"

        # Detect LLM dtype and hidden size
        self.llm_dtype = next(llm_model.parameters()).dtype
        self.llm_hidden_size = llm_model.config.hidden_size  # 2048 for internlm/internlm2_5-1_8b
        print(f"  LLM dtype: {self.llm_dtype}")
        print(f"  LLM hidden_size: {self.llm_hidden_size}")
        print(f"  Fusion mode: {self.fusion_mode} (K={self.top_k_images})")

        # ============================================
        # APPLY LoRA TO LLM
        # ============================================
        lora_config = LoraConfig(
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=["q_proj", "v_proj"],
            bias="none",
            task_type=TaskType.CAUSAL_LM,
        )
        self.llm = get_peft_model(llm_model, lora_config)
        self.llm.print_trainable_parameters()

        # Needed so gradients flow through inputs_embeds to the projector
        self.llm.enable_input_require_grads()

        # Text Embeddings layer from LLM (frozen)
        self.text_embeddings = self.llm.get_input_embeddings()
        for param in self.text_embeddings.parameters():
            param.requires_grad = False

        # Freeze vision encoder
        for param in self.vision_encoder.parameters():
            param.requires_grad = False

        # ============================================
        # ASPECT QUERY CROSS-ATTENTION HEAD (trainable)
        # ============================================
        self.aspect_queries = nn.Parameter(
            torch.randn(num_aspects, self.llm_hidden_size) * 0.02
        )
        self.aspect_attn = nn.MultiheadAttention(
            self.llm_hidden_size,
            num_heads=8,
            batch_first=True,
            dropout=0.1,
        )
        self.aspect_norm = nn.LayerNorm(self.llm_hidden_size)

        # ============================================
        # TWO-STAGE HEADS (trainable, float32)
        # ============================================
        # Stage 1: Presence — is aspect present? (binary)
        self.presence_head = nn.Linear(self.llm_hidden_size, 1)
        nn.init.zeros_(self.presence_head.weight)
        nn.init.zeros_(self.presence_head.bias)

        # Stage 2: Sentiment — Neg(0) / Neu(1) / Pos(2) (only for present aspects)
        self.sentiment_head = nn.Linear(self.llm_hidden_size, num_sentiment_classes)
        nn.init.xavier_uniform_(self.sentiment_head.weight)
        nn.init.zeros_(self.sentiment_head.bias)

        print(f"  Aspect queries: {num_aspects} x {self.llm_hidden_size}")
        print(f"  Presence head: {self.llm_hidden_size} -> 1 (per aspect)")
        print(f"  Sentiment head: {self.llm_hidden_size} -> {num_sentiment_classes} (Neg/Neu/Pos)")

        # Dtype summary for debugging
        print(f"\n  [DTYPE SUMMARY]")
        print(f"    Vision Encoder:  {self.vision_encoder.torch_dtype}")
        print(f"    MLPProjector:    {next(self.projector.parameters()).dtype}")
        print(f"    LLM (decoder):   {self.llm_dtype}")
        print(f"    Text Embeddings: {next(self.text_embeddings.parameters()).dtype}")
        print(f"    Presence Head:   {next(self.presence_head.parameters()).dtype}")
        print(f"    Sentiment Head:  {next(self.sentiment_head.parameters()).dtype}")
        ve_dt = self.vision_encoder.torch_dtype
        if ve_dt != self.llm_dtype:
            print(f"    [WARN] Vision ({ve_dt}) != LLM ({self.llm_dtype}). "
                  f"Ensure COMPUTE_DTYPE is set consistently.")

        # Keep raw backward() finite for smoke/integration tests and optimizer steps.
        self._grad_safety_handles = []
        self._register_grad_safety_hooks()
        self.last_image_weights = None  # debug cache

    def train(self, mode=True):
        super().train(mode)
        self.vision_encoder.eval()
        self.llm.train(mode)
        return self

    def _make_grad_safety_hook(self):
        def hook(grad: torch.Tensor) -> torch.Tensor:
            if grad is None:
                return grad
            if torch.isfinite(grad).all():
                return grad
            return torch.nan_to_num(grad, nan=0.0, posinf=0.0, neginf=0.0)
        return hook

    def _register_grad_safety_hooks(self):
        grad_hook = self._make_grad_safety_hook()
        for _, param in self.named_parameters():
            if param.requires_grad:
                self._grad_safety_handles.append(param.register_hook(grad_hook))

    def _check_nan(self, tensor: torch.Tensor, name: str) -> torch.Tensor:
        """Check for NaN/Inf and log a warning instead of silently replacing.
        Returns sanitized tensor. Use sparingly — only at module boundaries."""
        has_nan = torch.isnan(tensor).any().item()
        has_inf = torch.isinf(tensor).any().item()
        if has_nan or has_inf:
            finite_vals = tensor[torch.isfinite(tensor)]
            if finite_vals.numel() > 0:
                print(f"[NaN-DETECT] {name}: nan={has_nan}, inf={has_inf}, "
                      f"range=[{finite_vals.min().item():.4f}, {finite_vals.max().item():.4f}]")
            else:
                print(f"[NaN-DETECT] {name}: nan={has_nan}, inf={has_inf}, "
                      f"ALL values are NaN/Inf (no finite values) — likely dtype mismatch")
            tensor = torch.nan_to_num(tensor, nan=0.0, posinf=1e4, neginf=-1e4)
        return tensor

    def _masked_text_summary(self, text_lang_tokens: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        """Masked mean over text tokens -> [B, H]."""
        if attention_mask is None:
            return text_lang_tokens.float().mean(dim=1)

        mask = attention_mask.to(device=text_lang_tokens.device, dtype=torch.float32).unsqueeze(-1)  # [B, T, 1]
        denom = mask.sum(dim=1).clamp(min=1.0)
        return (text_lang_tokens.float() * mask).sum(dim=1) / denom

    def _get_combined_embeds(self, pixel_values, input_ids, attention_mask, image_counts=None):
        """Build combined embeddings with TOP-K image selection."""
        B = pixel_values.shape[0]

        # ====== TEXT PATH (needed early for image attention query) ======
        with torch.no_grad():
            text_lang_tokens = self.text_embeddings(input_ids.to(pixel_values.device))  # [B, T, H]

        # ====== IMAGE PATH ======
        if pixel_values.dim() == 5:
            # pixel_values: [B, M, 3, 256, 256]
            M = pixel_values.shape[1]
            P = self.vision_encoder.num_patches

            if image_counts is None:
                image_counts = torch.full((B,), M, device=pixel_values.device, dtype=torch.long)
            else:
                image_counts = image_counts.to(device=pixel_values.device, dtype=torch.long).clamp(min=0, max=M)

            valid_mask_2d = torch.arange(M, device=pixel_values.device).unsqueeze(0) < image_counts.unsqueeze(1)  # [B, M]
            valid_indices = valid_mask_2d.reshape(-1).nonzero(as_tuple=True)[0]

            pixel_flat_all = pixel_values.reshape(B * M, *pixel_values.shape[2:])
            pixel_flat_valid = pixel_flat_all[valid_indices] if valid_indices.numel() > 0 else pixel_flat_all[:0]

            with torch.no_grad():
                if pixel_flat_valid.numel() > 0:
                    img_tokens_valid = self.vision_encoder(pixel_flat_valid)  # [N_valid, P, 1024]
                else:
                    img_tokens_valid = torch.zeros(
                        0, P, self.vision_encoder.hidden_size,
                        dtype=self.vision_encoder.torch_dtype,
                        device=pixel_values.device,
                    )

            img_tokens_bmpv = torch.zeros(
                B * M, P, self.vision_encoder.hidden_size,
                dtype=img_tokens_valid.dtype,
                device=img_tokens_valid.device,
            )
            if valid_indices.numel() > 0:
                img_tokens_bmpv[valid_indices] = img_tokens_valid
            img_tokens_bmpv = img_tokens_bmpv.reshape(B, M, P, self.vision_encoder.hidden_size)  # [B, M, P, 1024]

            # Project each image independently to LLM space
            img_tokens_f32 = self._check_nan(img_tokens_bmpv.float(), "vision_output")
            img_lang_tokens_bmph = self.projector(
                img_tokens_f32.reshape(B * M, P, self.vision_encoder.hidden_size)
            ).reshape(B, M, P, self.llm_hidden_size)  # [B, M, P, H]
            img_lang_tokens_bmph = self._check_nan(img_lang_tokens_bmph, "projector_output")

            # ============================================
            # TOP-K IMAGE SELECTION (replaces weighted-sum)
            # ============================================
            # Text-guided scoring: cosine similarity between text summary and each image
            text_summary = self._check_nan(self._masked_text_summary(text_lang_tokens, attention_mask), "text_summary")  # [B, H]
            image_summary = self._check_nan(img_lang_tokens_bmph.float().mean(dim=2), "image_summary")  # [B, M, H]

            text_unit = torch.nn.functional.normalize(text_summary, p=2, dim=-1, eps=1e-6)
            image_unit = torch.nn.functional.normalize(image_summary, p=2, dim=-1, eps=1e-6)
            image_scores = (image_unit * text_unit.unsqueeze(1)).sum(dim=-1)  # [B, M]

            # Mask invalid images with very negative score
            image_scores = image_scores.masked_fill(~valid_mask_2d, -1e4)
            self.last_image_weights = image_scores.detach()  # debug cache (now scores, not weights)

            # Select top-K images per sample
            K = min(self.top_k_images, M)
            # For each sample, clamp K to the number of valid images
            actual_valid = image_counts.clamp(min=1)  # at least 1
            per_sample_k = torch.minimum(
                torch.full_like(actual_valid, K),
                actual_valid,
            )  # [B]

            # Use topk with the max K, then mask out extras
            _, topk_indices = image_scores.topk(K, dim=1)  # [B, K]

            # Gather patch tokens of selected images: [B, K, P, H]
            topk_exp = topk_indices[:, :, None, None].expand(-1, -1, P, self.llm_hidden_size)
            selected_img_tokens = torch.gather(img_lang_tokens_bmph, 1, topk_exp)  # [B, K, P, H]

            # Reshape to [B, K*P, H]
            img_lang_tokens = selected_img_tokens.reshape(B, K * P, self.llm_hidden_size)
            img_lang_tokens = self._check_nan(img_lang_tokens, "topk_img_tokens")

            # Build attention mask for image tokens: [B, K*P]
            # For samples with fewer than K valid images, mask out the padding
            img_attn = torch.ones(B, K * P, device=pixel_values.device, dtype=torch.long)
            for b in range(B):
                k_b = per_sample_k[b].item()
                if k_b < K:
                    # Zero out attention for padded image slots
                    img_attn[b, k_b * P:] = 0
        else:
            # Single-image path
            with torch.no_grad():
                img_tokens = self.vision_encoder(pixel_values)
            img_tokens_f32 = self._check_nan(img_tokens.float(), "vision_output")
            img_lang_tokens = self.projector(img_tokens_f32)  # [B, P, H]
            img_lang_tokens = self._check_nan(img_lang_tokens, "projector_output")
            self.last_image_weights = None
            img_attn = torch.ones(B, img_lang_tokens.size(1), device=img_lang_tokens.device, dtype=torch.long)

        # Cast BOTH to llm_dtype for decoder input
        img_lang_tokens = img_lang_tokens.to(dtype=self.llm_dtype)
        text_lang_tokens = text_lang_tokens.to(device=img_lang_tokens.device, dtype=self.llm_dtype)

        # ====== [image | text] concat ======
        combined_embeds = torch.cat([img_lang_tokens, text_lang_tokens], dim=1)

        # ====== Attention mask ======
        device = img_lang_tokens.device
        img_attn = img_attn.to(device=device, dtype=torch.long)

        if attention_mask is not None:
            text_attn = attention_mask.to(device=device, dtype=torch.long)
        else:
            text_attn = torch.ones(B, text_lang_tokens.size(1), device=device, dtype=torch.long)

        full_attention_mask = torch.cat([img_attn, text_attn], dim=1)
        return combined_embeds, full_attention_mask

    def _llm_hidden_states(self, combined_embeds: torch.Tensor, full_attention_mask: torch.Tensor) -> torch.Tensor:
        """Run the PEFT-wrapped LLM and return the FULL last-layer hidden states [B, L, H]."""
        combined_embeds = combined_embeds.to(dtype=self.llm_dtype)

        outputs = self.llm(
            inputs_embeds=combined_embeds,
            attention_mask=full_attention_mask,
            output_hidden_states=True,
            use_cache=False,
            return_dict=True,
        )
        hidden_states = outputs.hidden_states[-1]

        return self._check_nan(hidden_states, "llm_hidden_states")

    def forward(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor = None,
        image_counts: torch.Tensor = None,
    ) -> dict:
        B = pixel_values.shape[0]

        combined_embeds, full_attention_mask = self._get_combined_embeds(
            pixel_values, input_ids, attention_mask, image_counts
        )

        # ====== Get FULL hidden states from the LLM [B, L, H_llm] ======
        hidden_states = self._llm_hidden_states(combined_embeds, full_attention_mask)

        # ====== Aspect query cross-attention ======
        H = hidden_states.float()  # [B, L, H_llm] — float32 for head stability
        Q = self.aspect_queries.unsqueeze(0).expand(B, -1, -1)  # [B, 6, H_llm]

        # key_padding_mask: True = IGNORE this position
        key_padding_mask = ~full_attention_mask.bool()  # [B, L]

        Z, _ = self.aspect_attn(Q, H, H, key_padding_mask=key_padding_mask)  # [B, 6, H_llm]
        Z = self.aspect_norm(Z)  # [B, 6, H_llm]
        Z = self._check_nan(Z, "aspect_Z")
        Z = torch.clamp(Z, -1e3, 1e3)

        # ====== TWO-STAGE HEADS (float32) ======
        presence_logits = self.presence_head(Z).squeeze(-1)   # [B, 6]
        sentiment_logits = self.sentiment_head(Z)              # [B, 6, 3]

        return {
            "presence_logits": presence_logits,
            "sentiment_logits": sentiment_logits,
        }

    def get_trainable_params(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total, trainable


# Tao model (voi LoRA cho LLM)
sentiment_model = MultimodalSentimentModel(
    vision_encoder=vision_encoder,
    projector=projector,
    llm_model=model,
    tokenizer=tokenizer,
    num_aspects=NUM_ASPECTS,
    num_sentiment_classes=3,  # Neg/Neu/Pos (NOT 4!)
    top_k_images=TOP_K_IMAGES,
    lora_r=16,
    lora_alpha=16,
    lora_dropout=0.0,
).to(device)

total_params, trainable_params = sentiment_model.get_trainable_params()

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

In [ ]:
# TRAINING CONFIGURATION — Two-Stage Head
# ============================================

from torch.optim import AdamW
import torch.nn.functional as F
import math

# Hyperparameters
LEARNING_RATE = 2e-5      # For projector + heads + aspect queries
LORA_LR = 1e-6            # Much smaller LR for LoRA
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 10
WARMUP_PROPORTION = 0.1
GRADIENT_ACCUMULATION_STEPS = 2

# ============================================
# NO GRADIENT CHECKPOINTING
# ============================================
sentiment_model.llm.gradient_checkpointing_disable()
print("[OK] Gradient checkpointing DISABLED")

# ============================================
# NO CLASS WEIGHTS — two-stage loss is naturally balanced
# L_presence = BCEWithLogitsLoss (mean over all aspect slots)
# L_sentiment = CrossEntropyLoss (mean over present aspects only)
# λ₁ = λ₂ = 1.0
# ============================================
LAMBDA_PRESENCE = 1.0
LAMBDA_SENTIMENT = 1.0
print(f"  Loss: L = {LAMBDA_PRESENCE}*BCEWithLogits(presence) + {LAMBDA_SENTIMENT}*CrossEntropy(sentiment)")
print(f"  NO class_weights — two-stage naturally handles None vs sentiment")

# ============================================
# SEPARATE PARAM GROUPS
# ============================================
head_named_params = []
lora_named_params = []

for name, param in sentiment_model.named_parameters():
    if not param.requires_grad:
        continue
    if 'lora_' in name:
        lora_named_params.append((name, param))
    else:
        head_named_params.append((name, param))

head_params = [param for _, param in head_named_params]
lora_params = [param for _, param in lora_named_params]

captured_trainable = sum(p.numel() for p in head_params) + sum(p.numel() for p in lora_params)
all_trainable = sum(p.numel() for p in sentiment_model.parameters() if p.requires_grad)

print(f"  Head params (LR={LEARNING_RATE}): {sum(p.numel() for p in head_params):,}")
print(f"  LoRA params (LR={LORA_LR}): {sum(p.numel() for p in lora_params):,}")
print(f"  Captured trainable params: {captured_trainable:,}/{all_trainable:,}")

if captured_trainable != all_trainable:
    raise RuntimeError("Some trainable parameters were not assigned to optimizer groups")

optimizer = AdamW([
    {"params": head_params, "lr": LEARNING_RATE},
    {"params": lora_params, "lr": LORA_LR},
], weight_decay=WEIGHT_DECAY)

# Scheduler (count real optimizer steps, not raw batches)
steps_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_PROPORTION)

def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

class LazyLambdaScheduler:
    def __init__(self, optimizer, lr_lambda):
        self.optimizer = optimizer
        self.lr_lambda = lr_lambda
        self.base_lrs = []
        for group in optimizer.param_groups:
            base_lr = group['lr']
            group.setdefault('initial_lr', base_lr)
            self.base_lrs.append(base_lr)
        self._last_lr = list(self.base_lrs)
        self.last_epoch = -1
        self.started = False

    def _apply(self, step_idx):
        scale = float(self.lr_lambda(step_idx))
        self._last_lr = []
        for base_lr, group in zip(self.base_lrs, self.optimizer.param_groups):
            lr = base_lr * scale
            group['lr'] = lr
            self._last_lr.append(lr)
        self.last_epoch = step_idx

    def start(self):
        if self.started:
            return
        self.started = True
        self._apply(0)

    def step(self):
        if not self.started:
            self.start()
            return
        self._apply(self.last_epoch + 1)

    def get_last_lr(self):
        return list(self._last_lr)

scheduler = LazyLambdaScheduler(optimizer, lr_lambda)

print(f"\nTraining Configuration:")
print(f"  Head LR: {LEARNING_RATE}, LoRA LR: {LORA_LR}")
print(f"  Epochs: {NUM_EPOCHS}, Steps/Epoch: {steps_per_epoch}, Total Steps: {total_steps}, Warmup: {warmup_steps}")
print(f"  Gradient Accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"  Gradient Checkpointing: DISABLED")
print(f"  Early Stopping: enabled (patience will be set in training loop)")

In [ ]:
# ============================================
# PRE-TRAIN FORWARD SANITY CHECK
# ============================================

sentiment_model.eval()
batch0 = next(iter(train_loader))
with torch.no_grad():
    pixel_values = batch0['pixel_values'].to(device, dtype=COMPUTE_DTYPE)
    image_counts = batch0['image_counts'].to(device)
    input_ids = batch0['input_ids'].to(device)
    attention_mask = batch0['attention_mask'].to(device)

    print(f'[DTYPE CHECK] pixel_values={pixel_values.dtype}, '
          f'vision_encoder={vision_encoder.torch_dtype}, '
          f'llm={sentiment_model.llm_dtype}, '
          f'projector={next(sentiment_model.projector.parameters()).dtype}')

    out0 = sentiment_model(pixel_values, input_ids, attention_mask, image_counts)

presence_logits0 = out0['presence_logits']   # [B, 6]
sentiment_logits0 = out0['sentiment_logits']  # [B, 6, 3]
print(f'tokenizer.padding_side={tokenizer.padding_side}, pad_token_id={tokenizer.pad_token_id}')
print(f'image_counts={image_counts.tolist()}')
print(f'text valid lengths={attention_mask.sum(dim=1).tolist()}')
print(f'presence_logits shape={presence_logits0.shape}')
print(f'presence_logits finite={torch.isfinite(presence_logits0).all().item()} '
      f'range=[{presence_logits0.min().item():.4f}, {presence_logits0.max().item():.4f}]')
print(f'sentiment_logits shape={sentiment_logits0.shape}')
print(f'sentiment_logits finite={torch.isfinite(sentiment_logits0).all().item()} '
      f'range=[{sentiment_logits0.min().item():.4f}, {sentiment_logits0.max().item():.4f}]')

# Show predicted labels (combine presence + sentiment)
pred_presence = (presence_logits0 > 0).long()
pred_sentiment = sentiment_logits0.argmax(dim=-1) + 1  # 0->1(Neg), 1->2(Neu), 2->3(Pos)
pred_labels = pred_presence * pred_sentiment
print(f'predictions (combined): {pred_labels.tolist()}')

sentiment_model.train()


In [ ]:
# ============================================
# TRAINING LOOP — Two-Stage Head
# ============================================

def compute_loss(outputs, labels, device):
    """
    Two-stage loss for aspect-query model.

    Args:
        outputs: dict with 'presence_logits' [B, 6] and 'sentiment_logits' [B, 6, 3]
        labels: [B, 6] long, values 0-3  (0=None, 1=Neg, 2=Neu, 3=Pos)
        device: torch device

    Returns:
        total_loss: scalar
        loss_presence: scalar (for logging)
        loss_sentiment: scalar (for logging)
    """
    presence_logits = outputs["presence_logits"]   # [B, 6]
    sentiment_logits = outputs["sentiment_logits"]  # [B, 6, 3]

    # Guard non-finite logits
    if not torch.isfinite(presence_logits).all() or not torch.isfinite(sentiment_logits).all():
        nan_loss = torch.tensor(float('nan'), device=device, dtype=torch.float32)
        return nan_loss, nan_loss, nan_loss

    # ====== Stage 1: Presence loss ======
    # Binary: 1 if aspect is present (label > 0), else 0
    presence_labels = (labels > 0).float()  # [B, 6]
    L_presence = F.binary_cross_entropy_with_logits(
        presence_logits.float(), presence_labels,
        reduction='mean',  # mean over all B*6 aspect slots
    )

    # ====== Stage 2: Sentiment loss (only for present aspects) ======
    present_mask = (labels > 0)  # [B, 6] bool
    if present_mask.any():
        sent_logits_flat = sentiment_logits[present_mask]       # [N_present, 3]
        sent_labels_flat = (labels[present_mask] - 1).long()    # shift: 1->0, 2->1, 3->2
        L_sentiment = F.cross_entropy(
            sent_logits_flat.float(), sent_labels_flat,
            reduction='mean',  # mean over present aspects only
        )
    else:
        L_sentiment = torch.tensor(0.0, device=device, dtype=torch.float32)

    # ====== Combined loss ======
    total_loss = LAMBDA_PRESENCE * L_presence + LAMBDA_SENTIMENT * L_sentiment

    return total_loss, L_presence.detach(), L_sentiment.detach()


def _sanitize_grads(model):
    """Check for NaN/Inf gradients, log warning, and zero them out."""
    count = 0
    for name, p in model.named_parameters():
        if p.grad is not None:
            bad = torch.isnan(p.grad) | torch.isinf(p.grad)
            if bad.any():
                pct = bad.float().mean().item() * 100
                print(f"[WARN-GRAD] {name}: {pct:.1f}% NaN/Inf grads -> zeroed")
                p.grad[bad] = 0.0
                count += 1
    return count


def _optimizer_step(model, optimizer, scheduler):
    """Sanitize, clip, step, zero_grad. Returns number of sanitized params."""
    n_fixed = _sanitize_grads(model)
    if head_params:
        torch.nn.utils.clip_grad_norm_(head_params, max_norm=1.0)
    if lora_params:
        torch.nn.utils.clip_grad_norm_(lora_params, max_norm=0.3)
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad(set_to_none=True)
    return n_fixed


def train_epoch(model, dataloader, optimizer, scheduler,
                device, gradient_accumulation_steps=1):
    """Train 1 epoch with two-stage loss."""
    model.train()
    total_loss = 0.0
    total_presence_loss = 0.0
    total_sentiment_loss = 0.0
    num_batches = 0
    nan_batches = 0
    sanitized_steps = 0
    optimizer_steps = 0
    valid_micro_steps = 0

    optimizer.zero_grad(set_to_none=True)
    scheduler.start()

    progress_bar = tqdm(dataloader, desc="Training")

    for step, batch in enumerate(progress_bar):
        pixel_values = batch["pixel_values"].to(device, dtype=COMPUTE_DTYPE)
        image_counts = batch["image_counts"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(pixel_values, input_ids, attention_mask, image_counts)
        loss, l_pres, l_sent = compute_loss(outputs, labels, device)

        # Diagnostic on first batch
        if step == 0:
            print(f"\n[DIAG] step 0: tokenizer.padding_side={tokenizer.padding_side}, pad_token_id={tokenizer.pad_token_id}")
            print(f"[DIAG] step 0: image_counts={image_counts.tolist()}")
            print(f"[DIAG] step 0: presence_logits range=[{outputs['presence_logits'].min().item():.4f}, {outputs['presence_logits'].max().item():.4f}]")
            print(f"[DIAG] step 0: sentiment_logits range=[{outputs['sentiment_logits'].min().item():.4f}, {outputs['sentiment_logits'].max().item():.4f}]")
            print(f"[DIAG] step 0: loss={loss.item():.4f} (presence={l_pres.item():.4f}, sentiment={l_sent.item():.4f})")

        # Skip non-finite loss
        if not torch.isfinite(loss):
            nan_batches += 1
            if nan_batches <= 3:
                print(f"\n[WARN] Non-finite loss at step {step}")
            continue

        scaled_loss = loss / gradient_accumulation_steps
        scaled_loss.backward()
        valid_micro_steps += 1

        total_loss += loss.item()
        total_presence_loss += l_pres.item()
        total_sentiment_loss += l_sent.item()
        num_batches += 1

        if valid_micro_steps % gradient_accumulation_steps == 0:
            n_fixed = _optimizer_step(model, optimizer, scheduler)
            if n_fixed > 0:
                sanitized_steps += 1
            optimizer_steps += 1

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "pres": f"{l_pres.item():.4f}",
            "sent": f"{l_sent.item():.4f}",
        })

    # Flush leftover valid micro-batches
    if valid_micro_steps % gradient_accumulation_steps != 0:
        n_fixed = _optimizer_step(model, optimizer, scheduler)
        if n_fixed > 0:
            sanitized_steps += 1
        optimizer_steps += 1

    if nan_batches > 0:
        print(f"[WARN] {nan_batches} batches skipped (non-finite loss)")
    if sanitized_steps > 0:
        print(f"[INFO] {sanitized_steps}/{optimizer_steps} optimizer steps had NaN grads (sanitized to 0)")

    if num_batches == 0:
        return float('nan'), float('nan'), float('nan')

    avg_loss = total_loss / num_batches
    avg_pres = total_presence_loss / num_batches
    avg_sent = total_sentiment_loss / num_batches
    return avg_loss, avg_pres, avg_sent


@torch.no_grad()
def validate(model, dataloader, device):
    """Validate model (Two-Stage Head). Returns loss info + macro-F1."""
    from sklearn.metrics import f1_score

    model.eval()
    total_loss = 0.0
    total_presence_loss = 0.0
    total_sentiment_loss = 0.0
    num_batches = 0

    all_true_labels = []
    all_pred_labels = []

    for batch in tqdm(dataloader, desc="Validating"):
        pixel_values = batch["pixel_values"].to(device, dtype=COMPUTE_DTYPE)
        image_counts = batch["image_counts"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(pixel_values, input_ids, attention_mask, image_counts)
        loss, l_pres, l_sent = compute_loss(outputs, labels, device)

        if not torch.isfinite(loss):
            continue

        total_loss += loss.item()
        total_presence_loss += l_pres.item()
        total_sentiment_loss += l_sent.item()
        num_batches += 1

        # Combine presence + sentiment predictions into 4-class labels
        presence_logits = outputs["presence_logits"]    # [B, 6]
        sentiment_logits = outputs["sentiment_logits"]  # [B, 6, 3]

        pred_presence = (presence_logits > 0).long()                   # [B, 6], 0 or 1
        pred_sentiment = sentiment_logits.argmax(dim=-1) + 1           # [B, 6], 1/2/3
        pred_combined = pred_presence * pred_sentiment                  # [B, 6], 0/1/2/3

        all_true_labels.append(labels.cpu())
        all_pred_labels.append(pred_combined.cpu())

    if len(all_true_labels) == 0:
        print("[WARN] No valid validation batches")
        return float('nan'), float('nan'), float('nan'), 0.0, None, None

    avg_loss = total_loss / num_batches
    avg_pres = total_presence_loss / num_batches
    avg_sent = total_sentiment_loss / num_batches

    all_true = torch.cat(all_true_labels, dim=0)   # [N, 6]
    all_pred = torch.cat(all_pred_labels, dim=0)   # [N, 6]

    # Compute macro-F1 (flatten all aspects together)
    true_flat = all_true.numpy().reshape(-1)
    pred_flat = all_pred.numpy().reshape(-1)
    macro_f1 = f1_score(true_flat, pred_flat, average='macro', zero_division=0)

    return avg_loss, avg_pres, avg_sent, macro_f1, all_pred, all_true

In [ ]:
# ============================================
# TRAIN TWO-STAGE MODEL (checkpoint by macro-F1)
# ============================================

import os
import math
os.makedirs(OUTPUT_DIR, exist_ok=True)

EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_MIN_DELTA = 1e-4

best_macro_f1 = -1.0
epochs_without_improvement = 0
train_losses = []
val_losses = []
val_f1_scores = []

print("Starting Two-Stage Training (aspect queries + presence/sentiment heads)...")
print(f"  Checkpoint selection: macro-F1 on dev set (NOT val_loss)")
print(f"  Early Stopping: patience={EARLY_STOPPING_PATIENCE}, min_delta={EARLY_STOPPING_MIN_DELTA}")
print(f"  Output dir: {OUTPUT_DIR}")
print("=" * 60)

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 30)

    train_loss, train_pres, train_sent = train_epoch(
        sentiment_model, train_loader, optimizer, scheduler,
        device, GRADIENT_ACCUMULATION_STEPS
    )
    train_losses.append(train_loss)

    val_loss, val_pres, val_sent, macro_f1, val_preds, val_labels = validate(
        sentiment_model, dev_loader, device
    )
    val_losses.append(val_loss)
    val_f1_scores.append(macro_f1)

    print(f"Train Loss: {train_loss:.4f} (presence={train_pres:.4f}, sentiment={train_sent:.4f})")
    print(f"Val Loss:   {val_loss:.4f} (presence={val_pres:.4f}, sentiment={val_sent:.4f})")
    print(f"Val Macro-F1: {macro_f1:.4f}")

    if not math.isfinite(val_loss) or val_preds is None:
        print("[WARNING] Validation produced no valid predictions; skipping checkpoint update.")
        epochs_without_improvement += 1
    elif macro_f1 > best_macro_f1 + EARLY_STOPPING_MIN_DELTA:
        best_macro_f1 = macro_f1
        epochs_without_improvement = 0

        from safetensors.torch import save_file
        trainable_param_names = {
            name for name, param in sentiment_model.named_parameters() if param.requires_grad
        }
        trainable_state = {
            name: tensor.detach().cpu()
            for name, tensor in sentiment_model.state_dict().items()
            if name in trainable_param_names
        }
        missing_trainables = sorted(trainable_param_names - set(trainable_state.keys()))
        if missing_trainables:
            raise RuntimeError(
                f"Missing trainable tensors in save state: {missing_trainables[:10]}"
            )

        save_file(trainable_state, BEST_MODEL_PATH)
        torch.save({
            'epoch': epoch,
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'macro_f1': macro_f1,
            'num_aspects': NUM_ASPECTS,
            'num_sentiment_classes': 3,
            'aspect_labels': ASPECT_LABELS,
            'class_labels': CLASS_LABELS,
            'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
            'trainable_param_names': sorted(trainable_param_names),
            'lora_config': {
                'r': 16,
                'lora_alpha': 16,
                'lora_dropout': 0.0,
                'target_modules': ['q_proj', 'v_proj'],
            },
            'architecture': 'two_stage_aspect_query',
            'top_k_images': TOP_K_IMAGES,
        }, TRAINING_INFO_PATH, _use_new_zipfile_serialization=True)
        print(f"[SAVED] Best model saved to {OUTPUT_DIR}/ ({len(trainable_state)} tensors, macro-F1={macro_f1:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"[EARLY STOP] No macro-F1 improvement for {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE} epochs")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"\n[EARLY STOP] Stopping training: no macro-F1 improvement for {EARLY_STOPPING_PATIENCE} consecutive epochs.")
        break

print("\n" + "=" * 60)
print("Training Complete!")
if best_macro_f1 >= 0:
    print(f"Best Macro-F1: {best_macro_f1:.4f}")
else:
    print("Best Macro-F1: not available")


In [ ]:
# ============================================
# PLOT TRAINING CURVES - Train vs Dev Loss
# ============================================

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

epochs_range = range(1, len(train_losses) + 1)

ax.plot(epochs_range, train_losses, 'b-o', label='Train Loss', linewidth=2)
ax.plot(epochs_range, val_losses, 'r-o', label='Dev Loss', linewidth=2)

for i, (t, v) in enumerate(zip(train_losses, val_losses)):
    ax.annotate(f"{t:.4f}", (i + 1, t), textcoords="offset points", xytext=(0, 10), fontsize=8, ha='center', color='blue')
    ax.annotate(f"{v:.4f}", (i + 1, v), textcoords="offset points", xytext=(0, -15), fontsize=8, ha='center', color='red')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Train vs Dev Loss (Two-Stage Aspect-Query)', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## 7. Evaluation



In [ ]:
# ============================================
# EVALUATE ON TEST SET (Two-Stage Aspect-Query)
# ============================================

from safetensors.torch import load_file
from sklearn.metrics import precision_score, recall_score, f1_score
import os

model_path = BEST_MODEL_PATH
info_path = TRAINING_INFO_PATH

if os.path.exists(model_path):
    trainable_state = load_file(model_path)
    current_trainable_names = {
        name for name, param in sentiment_model.named_parameters() if param.requires_grad
    }
    missing_in_ckpt = sorted(current_trainable_names - set(trainable_state.keys()))
    unexpected_in_ckpt = sorted(set(trainable_state.keys()) - set(sentiment_model.state_dict().keys()))

    if missing_in_ckpt:
        print(f"[WARNING] Missing trainable keys in checkpoint: {missing_in_ckpt[:10]}")
    if unexpected_in_ckpt:
        print(f"[WARNING] Unexpected checkpoint keys: {unexpected_in_ckpt[:10]}")

    trainable_state = {k: v.to(device) for k, v in trainable_state.items()}
    sentiment_model.load_state_dict(trainable_state, strict=False)
    print(f"Loaded {len(trainable_state)} trainable tensors from {model_path}")

    if os.path.exists(info_path):
        try:
            info = torch.load(info_path, map_location='cpu', weights_only=True)
        except TypeError:
            info = torch.load(info_path, map_location='cpu')
        print(f"Loaded best model from epoch {info['epoch'] + 1}")
        print(f"  Val loss: {info['val_loss']}")
        if 'macro_f1' in info:
            print(f"  Val macro-F1: {info['macro_f1']}")
    else:
        print(f"Loaded best model from {OUTPUT_DIR}/")
else:
    print(f"[WARNING] No saved model found in {OUTPUT_DIR}/!")

test_loss, test_pres, test_sent, test_macro_f1, test_preds, test_labels = validate(
    sentiment_model, test_loader, device
)

print(f"\nTest Loss: {test_loss:.4f} (presence={test_pres:.4f}, sentiment={test_sent:.4f})")

if test_preds is None:
    print("[WARNING] Test evaluation produced no valid predictions.")
else:
    y_true = test_labels.numpy().ravel()
    y_pred = test_preds.numpy().ravel()

    macro_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    macro_recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    print("\n" + "=" * 70)
    print("TWO-STAGE EVALUATION RESULTS")
    print("=" * 70)
    print(f"Macro Precision: {macro_precision:.4f}")
    print(f"Macro Recall:    {macro_recall:.4f}")
    print(f"Macro F1:        {macro_f1:.4f}")
    print(f"Validate Macro-F1: {test_macro_f1:.4f}")


---
## 8. Inference



In [ ]:
# ============================================
# INFERENCE FUNCTIONS - Two-Stage Aspect-Query
# ============================================

def _combine_two_stage_predictions(presence_logits: torch.Tensor, sentiment_logits: torch.Tensor) -> torch.Tensor:
    """Convert two-stage outputs into 4-class per-aspect predictions."""
    pred_presence = (presence_logits > 0).long()
    pred_sentiment = sentiment_logits.argmax(dim=-1) + 1
    return pred_presence * pred_sentiment


def _two_stage_class_probabilities(presence_logits: torch.Tensor, sentiment_logits: torch.Tensor):
    """Build class probabilities for [None, Negative, Neutral, Positive]."""
    presence_probs = torch.sigmoid(presence_logits.float())
    sentiment_probs = torch.softmax(sentiment_logits.float(), dim=-1)

    combined_probs = torch.zeros(
        *presence_logits.shape,
        NUM_CLASSES,
        device=presence_logits.device,
        dtype=torch.float32,
    )
    combined_probs[..., 0] = 1.0 - presence_probs
    combined_probs[..., 1:] = presence_probs.unsqueeze(-1) * sentiment_probs
    return combined_probs, presence_probs, sentiment_probs


def _format_two_stage_result(presence_logits: torch.Tensor, sentiment_logits: torch.Tensor) -> dict:
    """Format one sample of two-stage outputs into notebook-friendly predictions."""
    combined_probs, presence_probs, sentiment_probs = _two_stage_class_probabilities(
        presence_logits, sentiment_logits
    )
    pred_classes = _combine_two_stage_predictions(presence_logits, sentiment_logits)

    combined_probs = combined_probs.cpu()
    presence_probs = presence_probs.cpu()
    sentiment_probs = sentiment_probs.cpu()
    pred_classes = pred_classes.cpu()

    detected_aspects = []
    aspect_sentiments = {}
    detailed = {}

    for a_idx in range(NUM_ASPECTS):
        aspect_name = ID2ASPECT[a_idx]
        pred_class = pred_classes[a_idx].item()
        pred_label = ID2CLASS[pred_class]

        detailed[aspect_name] = {
            'predicted_class': pred_class,
            'predicted_label': pred_label,
            'presence_probability': presence_probs[a_idx].item(),
            'probabilities': {
                ID2CLASS[c]: combined_probs[a_idx, c].item() for c in range(NUM_CLASSES)
            },
            'sentiment_probabilities': {
                'Negative': sentiment_probs[a_idx, 0].item(),
                'Neutral': sentiment_probs[a_idx, 1].item(),
                'Positive': sentiment_probs[a_idx, 2].item(),
            },
        }

        if pred_class > 0:
            detected_aspects.append(aspect_name)
            aspect_sentiments[aspect_name] = pred_label

    return {
        'detected_aspects': detected_aspects,
        'aspect_sentiments': aspect_sentiments,
        'detailed': detailed,
    }


def predict_aspect_sentiment(
    image_path: str,
    comment: str,
    model: MultimodalSentimentModel = sentiment_model,
) -> dict:
    """
    Single-image inference for the current two-stage aspect-query model.

    Pipeline:
      Text  -> Tokenizer -> text embeddings ----------?
                                                      ???> [image | text] -> InternLM2.5 + LoRA
      Image -> Swin V2 -> Projector -> image tokens --?    -> presence + sentiment heads
    """
    model.eval()

    transform = build_transform(IMAGE_SIZE)
    image = Image.open(image_path).convert('RGB')
    pixel_values = transform(image).unsqueeze(0).to(device, dtype=COMPUTE_DTYPE)

    text_inputs = tokenizer(
        [comment],
        padding=True,
        truncation=True,
        max_length=MAX_TEXT_LENGTH,
        return_tensors='pt',
    )
    input_ids = text_inputs.input_ids.to(device)
    attention_mask = text_inputs.attention_mask.to(device)

    with torch.no_grad():
        outputs = model(pixel_values, input_ids, attention_mask)

    return _format_two_stage_result(
        outputs['presence_logits'].squeeze(0),
        outputs['sentiment_logits'].squeeze(0),
    )


def predict_aspect_sentiment_batch(
    image_paths: list,
    comments: list,
    model: MultimodalSentimentModel = sentiment_model,
) -> list:
    """Batch inference for single-image samples using the current two-stage contract."""
    if len(image_paths) != len(comments):
        raise ValueError('image_paths and comments must have the same length')
    if len(image_paths) == 0:
        return []

    model.eval()
    transform = build_transform(IMAGE_SIZE)

    pixel_list = []
    for img_path in image_paths:
        image = Image.open(img_path).convert('RGB')
        pixel_list.append(transform(image))
    pixel_values = torch.stack(pixel_list).to(device, dtype=COMPUTE_DTYPE)

    text_inputs = tokenizer(
        comments,
        padding=True,
        truncation=True,
        max_length=MAX_TEXT_LENGTH,
        return_tensors='pt',
    )
    input_ids = text_inputs.input_ids.to(device)
    attention_mask = text_inputs.attention_mask.to(device)

    with torch.no_grad():
        outputs = model(pixel_values, input_ids, attention_mask)

    results = []
    for i in range(len(image_paths)):
        results.append(
            _format_two_stage_result(
                outputs['presence_logits'][i],
                outputs['sentiment_logits'][i],
            )
        )

    return results


In [ ]:
# ============================================
# DEMO INFERENCE - Two-Stage Aspect-Query Pipeline
# ============================================

import matplotlib.pyplot as plt

if len(test_dataset.samples) == 0:
    raise RuntimeError('test_dataset has no valid samples for demo inference.')

demo_sample = test_dataset.samples[0]
demo_image_path = demo_sample['image_paths'][0]
demo_comment = demo_sample['comment']
demo_true_labels = demo_sample['raw_labels']

print(f"Comment: {demo_comment}")
print(f"True labels: {demo_true_labels}")
print(f"Available images: {len(demo_sample['image_paths'])}")
print()

img = Image.open(demo_image_path)
plt.figure(figsize=(6, 4))
plt.imshow(img)
plt.axis('off')
plt.title(f"True: {demo_true_labels}")
plt.tight_layout()
plt.show()

result = predict_aspect_sentiment(demo_image_path, demo_comment)

print('=' * 60)
print('PREDICTION (Two-Stage Aspect-Query)')
print('=' * 60)
print(f"\n  Detected Aspects: {result['detected_aspects']}")
print(f"\n  Aspect-Sentiments:")
for aspect, sentiment in result['aspect_sentiments'].items():
    print(f"    {aspect}: {sentiment}")

print(f"\n  Detailed Probabilities:")
for aspect, info in result['detailed'].items():
    marker = '>>>' if info['predicted_class'] > 0 else '   '
    probs_str = ', '.join([f"{c}: {p:.3f}" for c, p in info['probabilities'].items()])
    print(f"  {marker} {aspect:<15} -> [{probs_str}] -> {info['predicted_label']}")
print('=' * 60)
